# Mediana e P90 da resolução informada

Tutorial complementar ao script 09. A mediana observada é 22,1 horas; o P90 é 381,55 horas, sobre 23.362 chamados elegíveis.

## Objetivo
Entender o tempo típico e a parte mais lenta da distribuição, com população e exclusões explícitas. Não é avaliação de SLA ou de produtividade.

## Preparação
Execute a partir da raiz do repositório ou da pasta notebooks. Requer pandas e o CSV local produzido pela etapa 06. O SQL de referência está em sql/04_indicadores_duracao.sql; sua saída foi salva em docs/duracao-sql.json após execução no Supabase.

Fonte: [UCI, Amaral et al., 2018](https://doi.org/10.24432/C57S4H), CC BY 4.0. Uma linha por chamado. Resoluções informadas de 29/02/2016 a 17/02/2017; sem fuso conhecido. Não usamos o calendário de fila como filtro de duração.

In [1]:
from pathlib import Path
import runpy
import pandas as pd

raiz = Path.cwd()
if not (raiz / 'scripts').exists():
    raiz = raiz.parent
analise = runpy.run_path(str(raiz / 'scripts/09_indicadores_duracao.py'))

## Etapas
### 1. Recalcular a duração e definir quem entra
Campos de data conflitantes ou inválidos, ausência, duração negativa e estado final inadequado são motivos de exclusão desta medida. Os motivos têm precedência para não contar duas vezes o mesmo chamado.

In [2]:
base = analise['carregar']()
print(base['motivo'].value_counts().to_string())

motivo
elegivel         23362
sem_resolucao     1556


### 2. Calcular os percentis sobre chamados individuais
Mediana = quantil 0,50. P90 = quantil 0,90 com interpolação linear. O cenário sem sinalizados é uma análise de sensibilidade, não uma correção comprovada.

In [3]:
resultados = analise['calcular'](base)
tabela = pd.DataFrame(resultados)
print(tabela.loc[tabela.recorte.eq('geral'), ['cenario', 'n_total', 'n_elegivel', 'n_excluido', 'mediana_horas', 'p90_horas']].to_string(index=False))

        cenario  n_total  n_elegivel  n_excluido  mediana_horas  p90_horas
          todos    24918       23362        1556      22.100000 381.548333
sem_sinalizados    24912       23356        1556      22.091667 381.550000


### 3. Entender a interpolação
Com durações 0, 10, 20 e 100, a mediana fica entre 10 e 20; o P90 fica entre 20 e 100. Um percentil contínuo pode não ser um valor existente na amostra.

In [4]:
exemplo = pd.Series([0.0, 10.0, 20.0, 100.0])
print('Mediana:', analise['quantil'](exemplo, 0.5))
print('P90:', analise['quantil'](exemplo, 0.9))

Mediana: 15.0
P90: 76.00000000000001


## Conferências
Comparação dos dois cenários e quatro prioridades finais em cada cenário: contagens exatas; tolerância de 0,000001 hora para percentis. A conferência usa a saída SQL salva; não abre uma nova conexão. Se o banco mudar, execute o SQL novamente e atualize a evidência antes de reconciliar.

In [5]:
quantidade = analise['conferir'](resultados)
print(f'SQL e Python conferem em {quantidade} recortes.')

SQL e Python conferem em 10 recortes.


## Próximos passos
1. Explicar por que 1.556 chamados sem resolução informada não viram duração zero.
2. Explicar por que não se calcula o P90 geral pela média dos P90 de prioridades.
3. Calcular a idade dos chamados pendentes para complementar a duração dos resolvidos.

Os 270 casos de prioridade final crítica têm mediana de 80,24 horas; os 674 elegíveis de baixa prioridade têm 5,02 horas. Isso não prova falha de priorização: são populações distintas, com prioridade final e exclusões diferentes.

Validação deste arquivo: células Python executadas sequencialmente no ambiente do projeto, com saídas capturadas; estrutura JSON conferida. O motor Jupyter não foi executado, pois suas dependências não estão instaladas.